# 02 · Analysis  the three questions

- **Q1** How many physicians do we need for the after hours telephone clinic?
- **Q2** How effective are those after-hours telephone appointments?
- **Q3** What would make them more effective?

For Q2, "effective" is treated as: is it a good *access channel*? i.e. does it see patients,
get the job done, and use physician time well. That fits MAPMG's care from home goals, and it's
also all the data lets us measure (there's no cancellation, call length, or diagnosis info).

In [1]:
import pandas as pd
import numpy as np
from silver import load_clean

df = load_clean()

# A text label for each combination of phone/in-person and after-hours/daytime.
df["group"] = df["is_telephone"].map({True: "Telephone", False: "In-Person"}) + " / " + \
              df["is_after_hours"].map({True: "After-Hours", False: "Daytime"})

# The group we care about most: telephone appointments after 17:30.
after_hours_phone = df[df["is_telephone"] & df["is_after_hours"]]
print("After-hours telephone appointments:", len(after_hours_phone))

After-hours telephone appointments: 4383


## Q1 · How many physicians?

The clinic runs on 10-minute slots, so one physician can handle about 3 patients in any 30-minute
stretch. So the plan: find the busiest 30-minute window each evening, then divide by 3 to get the
number of physicians. We only count patients who actually showed up.

**First, is the "10-minute slots" assumption actually true?** Nobody told us the slot length — there's no
such field in the data. So let's check the appointment times themselves: do they line up on 10-minute marks,
and what's the gap between one appointment and the next for the same physician on the same evening?

In [2]:
# 1) Do appointment times sit on 10-minute marks? (minute mod 10 == 0 means yes)
on_the_grid = (after_hours_phone["appt_min_of_day"] % 10 == 0).mean() * 100
print("Appointments landing exactly on a 10-minute mark:", round(on_the_grid, 1), "%")

# 2) Gap between back-to-back appointments for the same physician on the same evening.
#    (The first appointment of each evening has no gap, so it's excluded automatically.)
ordered = after_hours_phone.sort_values(["appt_date", "PROVIDER_ID", "appt_min_of_day"])
ordered["gap_minutes"] = ordered.groupby(["appt_date", "PROVIDER_ID"])["appt_min_of_day"].diff()

gap_counts = ordered["gap_minutes"].value_counts()
gap_table = pd.DataFrame({
    "count": gap_counts,
    "percent": (gap_counts / gap_counts.sum() * 100).round(1),  # % of all gaps
})
print("\nGap between consecutive appointments (count and % of all gaps):")
gap_table.head(6)

Appointments landing exactly on a 10-minute mark: 99.6 %

Gap between consecutive appointments (count and % of all gaps):


,count,percent
gap_minutes,,
10.0,3125,81.7
20.0,387,10.1
40.0,130,3.4
30.0,106,2.8
50.0,40,1.0
60.0,21,0.5


Almost every appointment (~99.6%) sits on a 10-minute mark, and the most common gap between back-to-back
appointments is exactly 10 minutes (the bigger gaps — 20, 30, 40… — are just empty slots in between). So a
**10-minute booking grid** is well supported by the data, which is why we use "3 patients per 30 minutes".

*Caveat:* this is the **booking** grid, not the actual call length — we have no call-duration field. If
real calls run longer than 10 minutes, the true staffing need would be a bit higher than what we estimate.

### First, what does demand look like by day and hour?

In [3]:
demand = after_hours_phone.groupby(["day_of_week", "appt_hour"]).agg(
    booked=("is_completed", "size"),
    showed_up=("is_completed", "sum"),
)
demand

booked  showed_up
day_of_week appt_hour                   
Friday      17            151        138
            18            296        237
            19            303        200
            20             13          4
Monday      17            199        187
            18            353        303
            19            151         94
            20              9          9
Saturday    18            211        199
            19            218        201
            20            106         97
Sunday      18            191        176
            19            177        161
            20             82         77
            21              1          0
Thursday    17            162        131
            18            276        209
            19            238        137
Tuesday     17            198        178
            18            274        222
            19            133         80
Wednesday   17            156        134
            18            257        196
            19            228        164

### Now find the busiest 30-minute window each evening

In [4]:
# Keep only patients who showed up.
showed = after_hours_phone[after_hours_phone["is_completed"]].copy()

# Label each appointment with its 30-minute block (e.g. 1050, 1080, ...).
showed["block_30min"] = (showed["appt_min_of_day"] // 30) * 30

# Count appointments in each block, each day.
counts = showed.groupby(["appt_date", "is_weekend", "block_30min"]).size()

# The busiest block per day = the peak we have to staff for.
busiest_per_day = counts.groupby(["appt_date", "is_weekend"]).max()
busiest_per_day = busiest_per_day.reset_index(name="peak")
busiest_per_day.head()

,appt_date,is_weekend,peak
0,2009-12-01,False,9
1,2009-12-02,False,6
2,2009-12-03,False,6
3,2009-12-04,False,6
4,2009-12-05,True,4


### Turn the peak into a physician count (weekday vs weekend)

In [5]:
def physicians_needed(peaks):
    return pd.Series({
        "typical_peak (median)": round(peaks.median()),
        "busy_peak (95th pct)": round(peaks.quantile(0.95)),
        "highest_peak (max)": peaks.max(),
        "physicians_recommended": int(np.ceil(peaks.quantile(0.95) / 3)),
    })

weekday_peaks = busiest_per_day[~busiest_per_day["is_weekend"]]["peak"]
weekend_peaks = busiest_per_day[busiest_per_day["is_weekend"]]["peak"]

pd.DataFrame({
    "Weekday": physicians_needed(weekday_peaks),
    "Weekend": physicians_needed(weekend_peaks),
})

,Weekday,Weekend
typical_peak (median),6,5
busy_peak (95th pct),9,6
highest_peak (max),9,7
physicians_recommended,3,2


**Q1 answer: about 3 physicians on weekday evenings, 2 on weekends, plus 1 spare for surge.**
The busy (95th percentile) weekday window needs ~9 patients covered → 9 ÷ 3 = 3 physicians.
The busiest single hour is 6 PM.

## Q2 · How effective are after-hours telephone appointments?

### Reliability: how often do patients actually show up?

In [6]:
no_show_rate = df.groupby("group")["is_noshow"].mean() * 100
no_show_rate.round(1)

group
In-Person / After-Hours    10.0
In-Person / Daytime         9.7
Telephone / After-Hours    19.4
Telephone / Daytime        17.8
Name: is_noshow, dtype: float64

After-hours telephone no-show is about **19%**, roughly double the in-person rate (~10%).
So the weakness is the phone format itself, not the time of day.

### Productivity: how many patients per physician-hour, phone vs in-person?

In [7]:
after_hours = df[df["is_after_hours"]].copy()

# For each physician on each evening, how many patients did they complete and over how long?
sessions = after_hours.groupby(["PROVIDER_ID", "appt_date", "is_telephone"]).agg(
    completed=("is_completed", "sum"),
    first_min=("appt_min_of_day", "min"),
    last_min=("appt_min_of_day", "max"),
)
# Session length in hours (add 10 minutes so a single-slot session isn't zero).
sessions["hours"] = (sessions["last_min"] - sessions["first_min"] + 10) / 60
sessions["per_hour"] = sessions["completed"] / sessions["hours"]

sessions.reset_index().groupby("is_telephone")["per_hour"].median().round(1)

is_telephone
False    2.2
True     3.8
Name: per_hour, dtype: float64

Telephone gets through about **3.8 patients/hour vs 2.2 in-person** — clearly more efficient.

### Does the call solve the problem, or just push it to an in-person visit?
For every after-hours telephone appointment, check if the same patient came in for an in-person visit
in the next 3 days. If the call were just deferring care, the patients who *did* talk to a doctor would
come back **more** than the ones who no-showed. Let's see.

In [8]:
# The after-hours phone appointments, and whether the patient showed up.
calls = after_hours_phone[["PATIENT_ID", "appt_date", "is_completed"]].copy()

# All in-person visits, renamed so we can compare dates after joining.
in_person = df[~df["is_telephone"]][["PATIENT_ID", "appt_date"]]
in_person = in_person.rename(columns={"appt_date": "in_person_date"})

# Match each call to that patient's in-person visits.
joined = calls.merge(in_person, on="PATIENT_ID", how="left")
days_later = (joined["in_person_date"] - joined["appt_date"]).dt.days
joined["came_in_within_3_days"] = (days_later > 0) & (days_later <= 3)

# Did each call lead to an in-person visit within 3 days?
per_call = joined.groupby(["PATIENT_ID", "appt_date", "is_completed"])["came_in_within_3_days"].max()
per_call = per_call.reset_index()

result = per_call.groupby("is_completed")["came_in_within_3_days"].mean() * 100
result.round(1)

is_completed
False    17.3
True     16.7
Name: came_in_within_3_days, dtype: float64

Completed calls **16.7%** vs no-show calls **17.3%** — basically the same. Read this carefully, because
it cuts both ways:

- **What it rules out:** talking to the doctor doesn't *create* extra in-person visits, so the call isn't
  actively **deferring** care.
- **What it does NOT prove:** it does **not** show the call *resolves* the issue. The no-show group had no
  call at all, yet ~83% of them *also* don't come in within 3 days. So "didn't follow up" can't be credited
  to the call — those patients may have self-resolved, gone **elsewhere (e.g. an ER we can't see)**, or gone
  without care. The two rates being equal means the call's clinical effect is simply **unmeasurable** here.

**Consequence for Q3:** since completing a call vs no-showing barely changes the 3-day in-person rate,
reducing no-shows will **not** cut downstream in-person visits — so we justify no-show reduction on
**wasted-capacity and access** grounds (a no-show burns a reserved physician slot another patient could
have used), *not* on resolution. Proving the call is clinically effective needs ER/encounter linkage — the
top data gap.

### Is the channel growing? Telephone share of after-hours each month

In [9]:
after_hours = df[df["is_after_hours"]]
monthly = after_hours.groupby("appt_month").agg(
    after_hours_total=("is_telephone", "size"),
    telephone=("is_telephone", "sum"),
)
monthly["telephone_share_%"] = (monthly["telephone"] / monthly["after_hours_total"] * 100).round(1)
monthly

,after_hours_total,telephone,telephone_share_%
appt_month,,,
2009-12,1029,597,58.0
2010-01,1124,697,62.0
2010-02,1158,718,62.0
2010-03,1212,770,63.5
2010-04,1116,698,62.5
2010-05,1062,650,61.2
2010-06,401,253,63.1


Telephone is a steady ~60% of after-hours every month — it's the preferred way patients reach the clinic.

## Q3 · How to improve effectiveness?

### Idea 1: last-minute bookings are the ones that no-show
Group the appointments by how long before the slot they were booked.

In [10]:
def lead_bucket(minutes):
    if minutes < 30:  return "1. under 30 min"
    if minutes < 60:  return "2. 30-60 min"
    if minutes < 120: return "3. 1-2 hours"
    if minutes < 240: return "4. 2-4 hours"
    return "5. 4+ hours"

phone = after_hours_phone.copy()
phone["booked_ahead"] = phone["booking_lead_min"].apply(lead_bucket)

phone.groupby("booked_ahead").agg(
    appointments=("is_noshow", "size"),
    no_show_pct=("is_noshow", "mean"),
).assign(no_show_pct=lambda t: (t["no_show_pct"] * 100).round(1))

,appointments,no_show_pct
booked_ahead,,
1. under 30 min,406,23.6
2. 30-60 min,1984,21.8
3. 1-2 hours,1445,15.8
4. 2-4 hours,483,16.6
5. 4+ hours,65,18.5


Booked under 30 min ahead → ~25% no-show; booked 1–2 hours ahead → ~16%. Since the typical booking is
only ~1 hour before the slot, a "2 hours before" reminder is too late — better to send a confirmation
**the moment they book**.

### Idea 2: which evenings are worst? Move staff to match

In [11]:
by_day = after_hours_phone.groupby("day_of_week").agg(
    appointments=("is_noshow", "size"),
    no_show_pct=("is_noshow", "mean"),
)
by_day["no_show_pct"] = (by_day["no_show_pct"] * 100).round(1)
# Put days in week order
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day.reindex(order)

,appointments,no_show_pct
day_of_week,,
Monday,712,16.7
Tuesday,605,20.7
Wednesday,641,22.9
Thursday,676,29.4
Friday,763,24.1
Saturday,535,7.1
Sunday,451,8.2


Thursday is worst (~29%), weekends are best (~7–8%). Midweek evenings are where the no-shows pile up.

### Idea 3: some physicians have far better show rates than others

In [12]:
by_provider = after_hours_phone.groupby("PROVIDER_ID").agg(
    appointments=("is_noshow", "size"),
    no_show_pct=("is_noshow", "mean"),
)
by_provider["no_show_pct"] = (by_provider["no_show_pct"] * 100).round(1)

# Only look at physicians with enough appointments to be meaningful.
busy_providers = by_provider[by_provider["appointments"] >= 50].sort_values("no_show_pct")
print("Best and worst no-show rates:", busy_providers["no_show_pct"].min(), "to", busy_providers["no_show_pct"].max())
busy_providers.tail()  # the worst few

Best and worst no-show rates: 0.0 to 43.0


,appointments,no_show_pct
PROVIDER_ID,,
25326992,65,26.2
57772125,70,27.1
79864737,66,30.3
15412841,68,30.9
42133974,242,43.0


Show rates range from 0% to 43%. The busiest physician is also the worst — worth understanding why and sharing what the best ones do.

### Idea 4: slots are often left empty, so fill them before hiring

In [13]:
# For each physician-evening, how many slots did they use vs how wide was their session?
sessions = after_hours_phone.groupby(["appt_date", "PROVIDER_ID"]).agg(
    slots_used=("appt_min_of_day", "size"),
    first_min=("appt_min_of_day", "min"),
    last_min=("appt_min_of_day", "max"),
)
sessions["slots_available"] = (sessions["last_min"] - sessions["first_min"]) / 10 + 1
sessions["fill_rate"] = sessions["slots_used"] / sessions["slots_available"]
print("Median fill rate:", round(sessions["fill_rate"].median(), 2))

Median fill rate: 0.79


About 79% of slots in a session are used — so ~1 in 5 sit empty, on top of the 19% no-shows. Better to
overbook the busy midweek evenings than add physicians.

------------

In [14]:
df["APPOINTMENT_TIME"][(df["PROVIDER_ID"] == "37153613") & (df["appt_date"] == pd.Timestamp("2009-12-02"))]

39366    12/30/99 17:30:00
39367    12/30/99 17:40:00
39368    12/30/99 17:50:00
39369    12/30/99 18:00:00
39370    12/30/99 18:10:00
39371    12/30/99 18:30:00
39372    12/30/99 19:00:00
39373    12/30/99 19:20:00
39374    12/30/99 19:30:00
Name: APPOINTMENT_TIME, dtype: object